# Enclave Evaluation — Researcher

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@ai-safety.org` | Trusted execution environment (sealed memory) |
| **Model Owner** | `model_owner@ai-safety.org` | Owns the private language-model weights (Qwen) |
| **Benchmark Owner** | `benchmark_owner@ai-safety.org` | Owns the private Vietnamese-harm benchmark |
| **Researcher** | _you_ | Writes the eval code, submits the job, scores results off-enclave |

This notebook drives only the **Researcher** steps; the Model Owner and Benchmark Owner run their own notebooks in parallel.
Self-contained: the eval code and local scoring helpers are fetched from the public repo.

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import requests

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_ds

# This notebook is self-contained — no local checkout of blindfold needed. Code + benchmark
# are fetched from the public repo; weights come from Hugging Face. If the repo moves to the
# OpenMined org, change GITHUB_REPO. Pin GITHUB_REF to a commit/tag for a reproducible demo.
GITHUB_REPO = "khoaguin/blindfold"
GITHUB_REF = "main"
RAW = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}"

# Downloads land here — next to the notebook (in Colab: /content/...) so the fetched code +
# benchmark show up in the Files pane and can be opened live during the demo.
BASE = Path.cwd()


def fetch(repo_path: str, dest: Path) -> Path:
    # Download one file from the public repo into `dest`.
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RAW}/{repo_path}"
    resp = requests.get(url, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"could not fetch {url} (HTTP {resp.status_code}) — is {GITHUB_REPO} public yet?")
    dest.write_bytes(resp.content)
    return dest

In [ ]:
ENCLAVE_EMAIL         = "enclave@ai-safety.org"
MODEL_OWNER_EMAIL     = "model_owner@ai-safety.org"
BENCHMARK_OWNER_EMAIL = "benchmark_owner@ai-safety.org"
RESEARCHER_EMAIL      = "researcher@ai-safety.org"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Model Owner: {MODEL_OWNER_EMAIL}  |  Benchmark Owner: {BENCHMARK_OWNER_EMAIL}")

## Step 0 — Log in as Researcher

In [ ]:
researcher = login_ds(RESEARCHER_EMAIL)
print(f"  Researcher : {researcher.email}")

In [ ]:
# Optionally clear state for a fresh run
researcher._manager.delete_syftbox()
researcher._manager.peer_manager.write_own_version()

### Launch the enclave

Start the enclave service for `enclave@ai-safety.org` out-of-band. It runs the job
automatically once **both** owners approve — no notebook drives it.

## Step 1 — Peer with Model Owner, Benchmark Owner, and the Enclave

In [ ]:
researcher.add_peer(MODEL_OWNER_EMAIL)
researcher.add_peer(BENCHMARK_OWNER_EMAIL)
researcher.add_peer(ENCLAVE_EMAIL)
researcher.sync()
print("  Peer requests sent to Model Owner, Benchmark Owner, and Enclave")

### Step 1.1 — Wait for the Data Owners to approve the peer request

The Model Owner and Benchmark Owner each run their `approve_peer_request` cell. Re-run the cell
below until both appear as approved peers.

In [ ]:
researcher.sync()
researcher.peers

### Step 1.2 — Attest enclave's identity

In [ ]:
# Wait for the enclave to accept the peer request, then attest
researcher.attest_peer(ENCLAVE_EMAIL)

## Step 2 — Browse the mock datasets

The researcher sees only the mocks (stub model + 3 sample rows) — never the private files.

In [ ]:
# Run twice if no table appears on the first run
researcher.sync()
researcher.datasets.get_all()

In [ ]:
try:
    researcher.datasets.get_all()[0].private_config
except Exception:
    print("expected: the researcher has no access to private configs / weights / data")

## Step 3 — Submit the eval (a reviewable code folder)

The eval code is fetched into **`./to_submit/`** (visible in the Files pane), then submitted to the
enclave: `main.py` (entrypoint) + `audit_core.py` (harm-origin map + offline language detection) +
`record.py` (output schema). The scoring half (`post_run/`) stays on the researcher's machine and
never enters the enclave. Both owners review the whole folder before approving. The enclave only
*generates* raw responses — no verdict.

In [ ]:
JOB_NAME = "vn_harm_audit"

# Eval code → ./to_submit/ (visible). Only what main.py imports (benchmark.py is unused → skipped).
JOB_DIR = BASE / "to_submit"
for _f in ("main.py", "audit_core.py", "record.py"):
    fetch(f"code/researcher_code/to_submit/{_f}", JOB_DIR / _f)

# params.json carries this run's owner emails + dataset names; the notebook writes it (not fetched).
params = {
    "model_owner": MODEL_OWNER_EMAIL,
    "bench_owner": BENCHMARK_OWNER_EMAIL,
    "model_dataset": "model",
    "benchmark_dataset": "vn_harm_benchmark",
    "model_label": "qwen2.5-0.5b",
    "demo_per_category": None,  # None = audit the full hidden set
}
(JOB_DIR / "params.json").write_text(json.dumps(params, indent=2))
print(f"  job folder -> {JOB_DIR}")

In [ ]:
researcher.submit_python_job(
    ENCLAVE_EMAIL,
    str(JOB_DIR),
    JOB_NAME,
    datasets={
        MODEL_OWNER_EMAIL: ["model"],
        BENCHMARK_OWNER_EMAIL: ["vn_harm_benchmark"],
    },
    dependencies=["mlx-lm", "pydantic"],
    entrypoint="main.py",
    share_results_with_do=True,
)
researcher.sync()
print(f"  Job '{JOB_NAME}' submitted to the enclave ({ENCLAVE_EMAIL})")

## Step 4 — Wait for both owners to approve and the enclave to run

Each owner approves from their notebook; once both approve, the enclave runs the job and pushes
results back. Re-sync until status is `"done"`.

In [ ]:
researcher.sync()
researcher_job = researcher.jobs[JOB_NAME]
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done", researcher_job.status
assert len(researcher_job.output_paths) > 0

## Step 5 — Retrieve the raw results

The local scoring helpers (`post_run/`) are fetched into **`./post_run/`** — they run here, never in
the enclave. One folder per run: `./results/<model>_<datetime>/`. Step 6 writes scored outputs into
the **same** folder.

In [ ]:
# Local scoring helpers → ./post_run/ (visible). Used here in Step 5 and the optional Step 6.
POST_RUN = BASE / "post_run"
for _f in ("scoring.py", "result.py", "judge.py", "report.py"):
    fetch(f"code/researcher_code/post_run/{_f}", POST_RUN / _f)
sys.path.insert(0, str(POST_RUN))

from report import make_run_dir, write_records_csv

with open(researcher_job.output_paths[0]) as f:
    out = json.load(f)

run_dir = make_run_dir(BASE / "results", out["model"])
write_records_csv(out["records"], run_dir / "results.csv")
print(f"  {len(out['records'])} raw responses -> {run_dir / 'results.csv'}")

## Step 6 — (Optional) Score locally, off the enclave · EN-vs-VN gap

The enclave returned only **raw** responses. Scoring runs **off-enclave**: an LLM judge labels
each response `refused | comply | partial`, then we tally the **EN-vs-VN refusal gap** per harm
category into this run's folder. Judging stays out of the enclave so no API key or network egress
ever touches the private boundary.

> Needs the researcher's local deps (`langchain-anthropic`, `python-dotenv`) and `ANTHROPIC_API_KEY`
> in `.env`; falls back to a keyword classifier otherwise.

In [ ]:
from judge import score_records
from report import render_report, write_records_csv, write_scorecard_csv
from scoring import tally

scored = score_records(out["records"])
write_records_csv(scored, run_dir / "scored.csv")
write_scorecard_csv(tally(scored), run_dir / "scorecard.csv")
report_md = render_report(scored, out["model"])
(run_dir / "REPORT.md").write_text(report_md, encoding="utf-8")

print(report_md)
print(f"\n  saved -> {run_dir}/  (REPORT.md, scored.csv, scorecard.csv)")